In [0]:
import json
import time
from datetime import date, datetime, timedelta, timezone

import requests

CATALOG = "workspace"
LANDING = f"/Volumes/{CATALOG}/bronze/landing/sgs"

SERIES = {
    1: "dolar_venda",
    432: "selic_meta",
    433: "ipca_mensal",
}

# Parâmetros: carga histórica usa data_inicial; o job diário usa dias_retroativos
dbutils.widgets.text("data_inicial", "01/01/2020")
dbutils.widgets.text("dias_retroativos", "")
dbutils.widgets.text("data_final", date.today().strftime("%d/%m/%Y"))

dias = dbutils.widgets.get("dias_retroativos").strip()
if dias:
    data_inicial = (date.today() - timedelta(days=int(dias))).strftime("%d/%m/%Y")
else:
    data_inicial = dbutils.widgets.get("data_inicial")
data_final = dbutils.widgets.get("data_final")


def buscar_serie(codigo: int, data_inicial: str, data_final: str) -> list[dict]:
    """Busca uma série do SGS/BCB com retry e backoff exponencial."""
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
    params = {"formato": "json", "dataInicial": data_inicial, "dataFinal": data_final}
    for tentativa in range(3):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 404:  # sem dados no período
                return []
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            if tentativa == 2:
                raise
            print(f"Série {codigo}: falha ({e}), tentando de novo...")
            time.sleep(2 ** tentativa)


extraido_em = datetime.now(timezone.utc)

for codigo, nome in SERIES.items():
    registros = buscar_serie(codigo, data_inicial, data_final)
    if not registros:
        print(f"{nome}: nenhum registro entre {data_inicial} e {data_final}")
        continue

    # JSON Lines: um registro por linha, com metadados de extração
    linhas = "\n".join(
        json.dumps({
            "serie_codigo": codigo,
            "serie_nome": nome,
            "extraido_em": extraido_em.isoformat(),
            **reg,
        })
        for reg in registros
    )

    caminho = f"{LANDING}/{nome}/{extraido_em:%Y%m%d_%H%M%S}.json"
    dbutils.fs.put(caminho, linhas, overwrite=True)
    print(f"{nome}: {len(registros)} registros -> {caminho}")